# SQLAlchemy: interagir avec une base de données en Flask 

On sait maintenant:
- créer une application Flask avec plusieurs routes
- définir les données à afficher via des paramètres d'URL
- gérer des templates pour chaque route, et combiner des templates pour former une vraie application

On peut donc construire une appli; il ne manque plus que **des données à afficher**. On va voir dans ce chapitre et dans les suivants comment **connecter une base de données SQL à une appli Flask**. 

**Le but, c'est que l'appli soit un *frontend* à votre base de données**: l'appli permettra de faire des opérations CRUD (*create-read-update-delete*) sur celle-ic.

--- 

# Installer SQLite et DbBrowser

On utilise:
- SQLite comme moteur de base de données SQL
- DBBrowser comme interface graphique (optionnel) pour découvrir la base

Normalement, SQLite au moins est déjà installé sur votre ordi.

**MacOS**:
```bash
brew install sqlite3
brew install db-browser-for-sqlite
```

**Linux**:
```bash
sudo apt install sqlite3
sudo apt install sqlitebrowser
```

---

# Le modèle de données

![db_schema](./img/db_schema.png)

Notre modèle de données est une version simplifiée du modèle de données de prod de la base Richelieu.

À noter:
- `place`, `theme` et `iconography` ont un champ `richelieu_url` qui renvoie vers la page de la ressource sur le site du Quartier Richelieu
- `place` décrit un lieu. Les informations spatiales sont décrites par `loc` (latitude/longitude du lieu) et `plot` (empreinte parcellaire du lieu: espace au sol d'un bâtiment, par exemple).
- dans `place` et `iconography`, les dates sont représentées par `date_lower` et `date_upper`:
    - si `date_lower` et `date_upper` sont définies, on a une tranche de date (*oeuvre créée entre 1845 et 1875*)
    - si seulement `date_lower` est définie, on a une date unique pour la ressource (*oeuvre créée en 1845*)
- toutes les ressources de la base proviennent de la NMF

---

# Connecter Flask à une base de données et faire une première requête: SQLAlchemy

## Connecter Flask à une base de données

### Le code 

In [2]:
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)
print(db)

/home/paul/Documents/cours/tnah_devapp/richelieu.db
<SQLAlchemy>


### Commentaire

#### Les librairies utilisées

- `flask_sqlalchemy` est "juste" un connecteur entre l'appli Flask et SQLAlchemy. Même si on importe SQLAlchemy depuis `flask_sqlalchemy`, le code SQLAlchemy qu'on verra est le même si on utilise SQLAlchemy tout seul.
- la documentation SQLAlchemy est [ici](https://www.sqlalchemy.org/). Le site n'est pas très joli, et la documentation est très complète mais assez vite technique. Ce [tutoriel](https://docs.sqlalchemy.org/en/20/orm/quickstart.html) couvrira beaucoup de choses vues plus tard.

#### Pour le code

- **on initialise son appli** Flask comme d'habitude:
    ```py
    app = Flask("Catalogue Richelieu")
    ```
- **`app.config` est un dictionnaire qui contient la configuration de notre** appli. Certaines variables sont prédéfinies. `SQLALCHEMY_DATABASE_URI` est l'URL de connexion à notre DB
    ```py
    app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
    ```
- **Pour s'y connecter à une DB SQLite avec SQLAlchemy**, on fait `sqlite:///chemin/vers/la/db`. (en SQLite, une base de données est un fichier, ce qui n'est pas le cas des autres variantes SQL).
- **`db` stockera notre base de données**

## Une première requête

`db` contient une référence à notre base de données. On peut donc directement faire des requêtes SQL sur notre `db`:


In [3]:
from sqlalchemy import text 

with app.app_context():
    result = db.session.execute(text("SELECT * FROM iconography;"))

print(result)

`with app.app_context()` est nécessaire car on ne travaille ici hors de l'application (on est pas dans une route). En général, on en a pas besoin. 

On fait une requête avec:

```py
result = db.session.execute(text("SELECT * FROM iconography;"))
```

- `db.session.execute(requete)` permet de lancer une requête.
- `db.session`, c'est la session (connexion à la base de données) de notre application Flask. (*voir l'explication détaillée plus bas*).
- `text()` est une fonction SQLAlchemy pour exécuter des requêtes textuelles 

*Pour le détail, `db.session`, c'est la `session` SQL utilisée par notre application Flask*. 
- *En SQL, un client est connecté à une base de données dans une session, qui est comme un espace de travail. Chaque client a sa propre session isolée du reste, dans laquelle il peut faire des requêtes*. 
- *Les requêtes SQL sont faites dans des `transactions`. Une transaction, c'est un ensemble d'opérations (insertions, modifications, suppressions) traitées comme un tout indivisible: soit toutes les opérations réussissent et sont validées (`commit`), soit une échoue et tout est annulé (`rollback`). Il pas d'état intermédiaire bancal. Une transaction peut être ouverte implicitement, mais on peut parfois en créer dans SQLAlchemy.*
- *TLDR: session > transaction > requêtes* 

In [4]:
print(result)

Quand on exécute le code, on ne voit pas les données, mais un: `CursorResult`. Un **curseur est un objet qui stocke les résultats** d'une requête en base de données. On peut faire [beaucoup de choses](https://docs.sqlalchemy.org/en/14/core/connections.html#sqlalchemy.engine.CursorResult) avec.

Deux méthodes à retenir:
- itérer sur `result`: 
    ```py
    for row in result
    ```
- utiliser `result.all()` pour avoir une liste de toutes les lignes:
    ```py
    rows = result.all()
    ```

In [5]:
# note: un curseur se ferme à chaque fois qu'on a accédé à tous les résultats => je fais une fonction pour relancer ma requête
def requete():
    with app.app_context():
        return db.session.execute(text("SELECT * FROM iconography;"))
    
# en bouclant
result = requete()
for row in result: 
    print("* ROW :", row)
    print("* TYPE ROW :", type(row))
    print("* ACCÉDER À UNE COLONNE :", row.title)

* ROW : (1, 'Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]', 'https://gallica.bnf.fr/iiif/ark:/12148/btv1b530124015/manifest.json', 'https://gallica.bnf.fr/iiif/ark:/12148/btv1b530124015/f1/full/1000/0/native.jpg', 'https://gallica.bnf.fr/ark:/12148/btv1b530124015', 'https://quartier-richelieu.inha.fr/iconographie/qr1703e89942a0c4fd08052e91956f67719', 1807, 1808, 'Bibliothèque nationale de France', 235)
* TYPE ROW : <class 'sqlalchemy.engine.row.Row'>
* ACCÉDER À UNE COLONNE : Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]
* ROW : (2, '[Galerie Colbert, rue Vivienne] : [Mai 1906] : [photographie] / [Atget]', 'https://gallica.bnf.fr/iiif/ark:/12148/btv1b10516371w/manifest.json', 'https://gallica.bnf.fr/iiif/ark:/12148/btv1b10516371w/f1/full/1000/0/native.jpg', 'https://gallica.bnf.fr/ark:/12148/btv1b10516371w', 'https://quartier-richelieu.i

In [6]:
# tous les résultats
result = requete()
all_rows = result.all() 
print("* ALL_ROWS :", all_rows)
print("* TYPE :", type(all_rows))
print("* NOMBRE DE LIGNES :", len(all_rows))
print("* TITRE DE LA 5e RESSOURCE ICONO: ", all_rows[4].title)

* ALL_ROWS : [(1, 'Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]', 'https://gallica.bnf.fr/iiif/ark:/12148/btv1b530124015/manifest.json', 'https://gallica.bnf.fr/iiif/ark:/12148/btv1b530124015/f1/full/1000/0/native.jpg', 'https://gallica.bnf.fr/ark:/12148/btv1b530124015', 'https://quartier-richelieu.inha.fr/iconographie/qr1703e89942a0c4fd08052e91956f67719', 1807, 1808, 'Bibliothèque nationale de France', 235), (2, '[Galerie Colbert, rue Vivienne] : [Mai 1906] : [photographie] / [Atget]', 'https://gallica.bnf.fr/iiif/ark:/12148/btv1b10516371w/manifest.json', 'https://gallica.bnf.fr/iiif/ark:/12148/btv1b10516371w/f1/full/1000/0/native.jpg', 'https://gallica.bnf.fr/ark:/12148/btv1b10516371w', 'https://quartier-richelieu.inha.fr/iconographie/qr1bd0eae99181a4c1aad1e56afc225a9f3', 1906, 1908, 'Bibliothèque nationale de France', 197), (3, '[Galerie Colbert, Rue Vivienne] : [Mai 1906] : [photographie] / [Atget]', 'https

## Une requête dans une route

Reprenons l'exemple au dessus pour faire la requête sur `iconographie` quand on va sur l'URL `/iconographie`. Comme ça, la requête sera exécutée chaque fois que l'on va sur `localhost:5000/iconographie`.

In [7]:
@app.route("/iconographie")
def icono_index():
    result = db.session.execute(text("SELECT * FROM iconography;"))
    icono_corpus = result.all()
    return render_template("pages/icono_index.html", app_name=APP_NAME, icono_corpus=icono_corpus)

On rajoute une template `icono_index.html`:

```html
{% extends "base.html" %}

{% block title_extra %}| Iconographie ({{ icono_corpus|length }} ressources){% endblock %}

{% block main_content %}
    <p>Il y a <b>{{ icono_corpus|length }} ressources iconographiques</b> 
        sur ce site.</p>

    <ul class="columns is-multiline mt-1">
        {% for item in icono_corpus %}
            <li class="column is-one-third">
                <div class="card border">
                    <span cardlass="card-header">   
                        <i class="card-header-title">{{ item.title }}</i>
                    </span>
                    <p class="card-content">
                        {% if item.date_lower %}
                            ({{ item.date_lower }}
                        {% endif %}-{% if item.date_upper %}
                            {{ item.date_upper }})
                        {% endif %}
                        {{ item.institution }}
                    </p>
                </div>
            </li>
        {% endfor %}        
    </ul>

{% endblock %}
```

Et voilà le travail ! 

> Lancez l'application `sqlalchemy_premiere_requete`:
> ```py
> python apps/s3/sqlalchemy_premiere_requete/main.py
> ```

(vous remarquerez dans le code que j'ai fait quelques autres changements. Notamment, `PATH_DB` et `APP_NAME` sont définies dans `constants.py`)


---

# Apparté: le typage en Python

Pour définir des tables, la **syntaxe SQLAlchemy demande d'utiliser des *type hints*, c'est à dire de savoir typer des variables en Python**.

## Comment donc typer une variable en Python: les bases

Vous le savez surement, **Python est un langage *dynamiquement typé***: pas besoin de dire qu'une variable est un `int`, une `str`... Python se charge de déterminer le type de chaque variable au moment où on exécute son programme.

Cependant, les types c'est bien pratique quand on développe des grosses appli: cela permet notamment de **documenter son code**.

In [8]:
# pas de type hint
a = 3
# avec type hint
a: int = 3

# et c'est une bonne pratique de faire des fonctions avec type hints, pour documenter ce que la fonction prend comme inputs et outputs
def summary(txt: str, n: int) -> str:
    if type(n) != int:
        print("summary ne fonctionne que si `n` est un `int` positif !")
        return ""
    if len(txt) > n and n>0:
        return txt[:n] + "[...]"
    return txt

print(summary("bonjour", 25))
print(summary("bonjour", 1))
print(summary("anticonstitutionnellement", 10))
print(summary("bonjour", "3"))

bonjour
b[...]
anticonsti[...]
summary ne fonctionne que si `n` est un `int` positif !



**La syntaxe est donc:**

```py
# dans une définition de variable
ma_variable: mon_type = ma_valeur
# exemple
film: int = "Twin Peaks"

# dans définition de une fontion
def ma_fonction(mon_argument: type_de_mon_argument) -> type_du_return:
    ...
def def summary(txt: str, n: int) -> str:
    ...
```

Dans le dernier exemple, on voit que **donner à une variable un mauvais type n'empêche pas un programme de se lancer**: on donne à `n` le mauvais type, la fonction s'exécute, même si votre IDE affiche peut-être un message d'erreur.

## Types paramétrables

`str`, `int` et `float` sont des types simples. `dict` et `list`, par exemple, sont complexes: une liste peut contenir plusieurs types de données. On spécifie ce que contient un autre type comme ça:

```py
from typing import List, Dict

# x est une liste de str
x: List[str]
# x est une liste de int
x: List[int]
# x est un dictionnaire où les clés sont des str et les valeurs des int
x: Dict[str, int]
# x est un dictionnaire où les clés sont des str et les valeurs des listes de int
x: Dict[str, List[int]]
```

## Types optionels 

Une variable peut aussi prendre une **valeur optionelle: `1 ou None`**. Cela se type avec **`Optional`**:

```py
from typing import List, Dict, Optional

# int ou None
x: Optional[int]
# str ou None
x: Optional[str]
# liste de dicts ou None
x: Optional[List[Dict]]
```

## Types variables

Et enfin, une variable peut **accepter différents types: `str ou int`**. Cela se type avec **`Union`**:

```py
from typing import List, Dict, Optiona, Union

# nombre entier ou décimal
x: Union[int, float]
# nombre entier ou liste de dict
x: Union[int, List[Dict]]
# nombre entier ou liste de dict ou liste vide (bon là ça devient compliqué)
x: Union[int, List[Optional[Dict]]]
```

## Note de compatibilité

La syntaxe pour les types s'est beaucoup simplifiée après Python 3.10: on peut utiliser `list` et `dict` au lieu de `List` et `Dict`, `int|None` au lieu de `Optional[int]`. J'utilise ici l'ancienne syntaxe pour éviter des bugs.

Revenons à nos moutons: modéliser une base de données avec SQLAlchemy.


---

# ORM et modèles: introduction

> apparté: est-ce que vous êtes à l'aise avec le concept de Classes et d'objets en Python ?

On sait maintenant faire du SQL en Python. Mais là où SQLAlchemy devient très utile, c'est qu'il nous permet de faire des **requêtes et de manipuler nos données directement en Python** !

C'est possible grâce à un **ORM (`object-relational mapping`)**. Un ORM permet de connecter:
- **le modèle de données relationnel** de votre base de données: tables, lignes, clés étrangères...
- **avec un modèle de données Python**, pour pouvoir manipuler vos données directement en Python (on voit comment plus bas)

On peut donc oublier SQL (enfin, presque) ! Plus de requêtes à écrire à la main, plus de jointures à faire. On définit notre modèle de données Python, l'ORM s'occupe de traduire notre code Python en requêtes SQL.

---

# Définir un modèle

Les modèles de données SQLAlchemy sont définis dans un style "déclaratif": on va déclarer un certain nombre d'objets qui sont des descriptions de la structure de notre base de données SQL. SQLAlchemy lira cette description et s'occupera de faire le lien SQL-Python.

Concrètement: 
- **chaque table SQL**, est représentée par une `class` Python
- **chaque colonne** est représentée par un attribut de cette classe
- **chaque ligne** d'une table est une instance de classe (= chaque ressource iconographique sera unb objet).

Voici comment on définit la table `iconography`:

In [9]:
from typing import Optional
from sqlalchemy.orm import Mapped, mapped_column

class Iconography(db.Model):
    __tablename__ = "Iconography"
    
    # comment lit-on chacun des attributs ci-dessous ?
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    # NOTE: il reste à définir la relation avec la table `author`, mais on verra comment faire plus tard !
    id_author = ...


Décomposons l'exemple au dessus.

## Définir une table

(désolé d'avance, c'est un peu théorique)

**Pour chaque table, on déclare un une classe qui hérite de `db.Model`** (`db` étant notre objet Flask-SQLAlchemy définit plus haut). `__tablename__` est optionnel et permet de définir explicitement le nom de notre table SQL. Définissons `Iconography`:

```py
# Iconography(db.Model) veut dire que notre classe est une sous-classe de db.Model, 
# donc que l'ORM SQLAlchemy la reconnaît comme une table 
class Iconography(db.Model):
    __tablename__ = "iconography"
```

## Définir une colonne

**Chaque colonne est une propriété de la classe**. Elle est définie par:
- **un type `Mapped`**, qui définit le type de données Python et si la colonne est nullable
- **une fonction `mapped_column`**. Celle-ci est souvent optionnelle et donne toutes les contraintes que `Mapped` ne peut décrire.

```py
from typing import Optional
from sqlalchemy.orm import Mapped, mapped_column

class Iconography(db.Model):
    __tablename__ = "iconography"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    source_url: Mapped[Optional[str]]
```

Revenons sur chaque colonne.

### `Iconography.id`: la clé primaire

```py
# colonne iconography.id
id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
```

#### `Mapped[int]`: le type de `Iconography.id`
- `Mapped`: type SQLAlchemy qui signifie qu'une variable **doit être traitée comme une colonne de table SQL**.
- `[int]` spécifie le type de la colonne: **c'est une colonne qui stocke des `int`**.

#### `mapped_column()`: les contraintes supplémentaires

`mapped_column(primary_key=True, autoincrement=True)` **définit toutes les contraintes supplémentaires** (qui me peuvent être définies dans `Mapped`)

- `mapped_column()` est une fonction SQLAlchemy qui **associe à un attribut de classe une colonne de table**.
- ses arguments sont les contraintes de la colonne: 
    - `primary_key=True`: la colonne contient une clé primaire
    - `autoincrement=True`: à chaque nouvel insert, l'ID est généré automatiquement

#### Équivalent SQL

```py
id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
```

Est équivalent à cette définition de colonne SQL:

```sql
-- not null et autoincrement sont implicites sur les clés primaires
id INTEGER PRIMARY KEY
```

### `Iconography.title`: le titre d'une ressource

```py
# colonne iconography.title
title: Mapped[str]
```

#### `Mapped[str]`: 

- `Mapped`: **c'est une colonne** de base de données
- `[str]`: **le type de la colonne**, c'est du texte


#### Pas de `mapped_column`

Il n'y a pas de `mapped_column`: toute la définition de la colonne est passée par le `Mapped`, et il n'y a **pas de contraintes supplémentaires** à faire passer par le `mapped_column`.

#### Équivalent SQL:

```py
title: Mapped[str]
```

est équivalent à :

```sql
title TEXT NOT NULL
```

### `Iconography.source_url`: l'URL d'origine de la ressource

La logique est la même:

```py
# colonne iconography.source_url
source_url: Mapped[Optional[str]]
```

- `source_url`: nom de colonne
- `Mapped[Optional[str]]`: `source_url` est une colonne SQL qui contient optionnellement des `str`
- équivalent SQL: `source_url TEXT`

### Pour résumer la syntaxe

Voici comment on définit une colonne via SQLAlchemy:

```py
class NomDeTable(db.Model):
    nom_de_propriete: Mapped[type_python] = mapped_column(autres_contraintes)
```

Pour résumer l'utilisation de `Mapped` / `mapped_column`:

| construct | rôle | quand utiliser |
|-----------|------|----------------|
| `Mapped[X]` | **type Python + inférence SQL** | **toujours** |
| `mapped_column()` | configurer les **détails SQL explicites** (type SQL, contraintes, clés, etc.) | **optionel la plupart du tempsst, obligatoire pour**: les clés primaires, les contraintes `UNIQUE`, les `ForeignKey` et certains types complexes (JSON) |


À partir de là, définissez la classe `Author` suivant ce modèle:

```sql
CREATE TABLE author (
    id INTEGER PRIMARY KEY, 
    author_name TEXT NOT NULL UNIQUE
);
```

In [10]:
# votre code ici...

Dans le prochain cours, on verra comment faire des jointures entre les tables !

--- 

# Requêtes via l'ORM et instances de classe

## Une première requête

Pour mieux comprendre la puissance des ORMs, regardons maintenant comment on peut requêter et manipuler ces deux tables !

Une fois notre modèle `Iconography` définit, la requête `SELECT * FROM iconography;` s'écrit:

In [11]:
with app.app_context():
    query = db.select(Iconography)
    icono_all = db.session.execute(query).scalars().all()

# comment interpréter ce résultat ?
print("* RÉSULTATS:", icono_all)
print("* 1er ITEM:", icono_all[0])
print("* TYPE:", type(icono_all[0]))

* RÉSULTATS: [<Iconography 1>, <Iconography 2>, <Iconography 3>, <Iconography 4>, <Iconography 5>, <Iconography 6>, <Iconography 7>, <Iconography 8>, <Iconography 9>, <Iconography 10>, <Iconography 11>, <Iconography 12>, <Iconography 13>, <Iconography 14>, <Iconography 15>, <Iconography 16>, <Iconography 17>, <Iconography 18>, <Iconography 19>, <Iconography 20>, <Iconography 21>, <Iconography 22>, <Iconography 23>, <Iconography 24>, <Iconography 25>, <Iconography 26>, <Iconography 27>, <Iconography 28>, <Iconography 29>, <Iconography 30>, <Iconography 31>, <Iconography 32>, <Iconography 33>, <Iconography 34>, <Iconography 35>, <Iconography 36>, <Iconography 37>, <Iconography 38>, <Iconography 39>, <Iconography 40>, <Iconography 41>, <Iconography 42>, <Iconography 43>, <Iconography 44>, <Iconography 45>, <Iconography 46>, <Iconography 47>, <Iconography 48>, <Iconography 49>, <Iconography 50>, <Iconography 51>, <Iconography 52>, <Iconography 53>, <Iconography 54>, <Iconography 55>, <Icon

### Explication de la requête

```py
query = db.select(Iconography)
icono_all = db.session.execute(query).scalars().all()
```

**Cette requête peut être divisée en 3 parties**:
- **la requête SQL** en elle-même: `db.select(Iconography)`
    - on reviendra sur la syntaxe plus bas, mais on remarque déjà que la syntaxe est très proche du SQL
- **l'éxécution de la requête**: `db.session.execute()`
    - la manière dont une requête est exécutée est exactement pareille à la requête textuelle vue au dessus: 
- **le traitement des résultats**: `.scalars().all()`
    - `.scalars()`: si on ne l'utilise pas, SQLAlchemy retourne *une liste de tuples d'objets `Iconography`*, plus dur à manipuler: `[(Icono1,), (Icono2,)]`
    - `.all()`: comme dans notre requête textuelle, `.all()` affiche tous les résultats dans une liste

**La bonne nouvelle**: seule la 1e partie change d'une requête à l'autre ! Donc même si la syntaxe fait un peut peur maintenant, on s'y habitue vite.

### Les résultats

**Pour les résultats, on voit que:**
- **`.all()` retourne une liste d'objets** (comme vu avec notre exemple en haut)
- **chaque item de la liste est un objet `Iconography`**. 

## Instances de classse

`<Iconography 1>, <Iconography 2>, ...`, ça veut dire quoi ? Si on remonte plus haut, on a dit que:

> chaque ligne d'une table est une instance de classe

Cela signifie que **chaque ligne de la table est une instance d'`Iconography`**, et qu'on peut manipuler chaque ligne de notre table comme un objet Python. Notamment, **chaque colonne est une propriété de l'objet**:

In [12]:
item_icono = icono_all[0]

print("Le titre est:", item_icono.title)
print("Ses dates sont:", item_icono.date_lower, item_icono.date_upper)
print("Son ID en base de données est:", item_icono.id)

Le titre est: Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]
Ses dates sont: 1807 1808
Son ID en base de données est: 1


In [13]:
# afficher tous les titres
for item_icono in icono_all:
    print(item_icono.title)

Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]
[Galerie Colbert, rue Vivienne] : [Mai 1906] : [photographie] / [Atget]
[Galerie Colbert, Rue Vivienne] : [Mai 1906] : [photographie] / [Atget]
Grand succès du Théatre Impérial de l'Opéra-Comique, Mignon... : Opéra en trois actes... paroles de MM. Carré et Jules Barbier..., musique de Ambroise Thomas.... Des mêmes auteurs, Hamlet..., partition, piano et chant..., Heugel et Cie, 2 bis, rue Vivienne, éditeurs pour la France et l'étranger. : [affiche] / [non identifié]
La Fille invisible, opéra-comique en 3 actes et 4 tableaux. Poème de Mrs de St Georges & Dupin. Partition orchestre et parties séparées [...]. Musique de A. Boieldieu. A Paris, Maison Boieldieu, Sylvain St. Etienne, sucesseur, rue Vivienne, 53, près le Boulevart [sic]. : [affiche] / Victor Coindre
La Fille invisible, opéra-comique en 3 actes et 4 tableaux. Poème de Mrs de St Georges & Dupin. Partition orc


---

# Requêtes `SELECT`: quelques éléments de syntaxe

Si on reprend cet exemple:

```py
query = db.select(Iconography)
icono_all = db.session.execute(query).scalars().all()
```

On va maintenant voir comment on construit notre `query`, et comment on peut faire des requêtes plus intéressantes

## `db.select()`: faire une requête `SELECT`

`db.select()` permet de produire une **requête select sur la table spécifiée en argument**. La syntaxe est:

```py
db.select(NomDeTable)
```

## `db.get()`: récupérer un élément par son `id`

Un `get` est un type particulier de requête pour sélectionner un objet par son `id`. La requête:

```sql
SELECT * FROM iconography WHERE iconography.id = 41
```

In [14]:
with app.app_context():
    icono_item = db.session.get(Iconography, 41)
print(icono_item)

<Iconography 41>


**La syntaxe** est donc:

```py
db.session.get(NomDeTable, id)
```

In [15]:
with app.app_context():
    icono_item = db.session.get(Iconography, 41)
print(icono_item)
print(icono_item.title)

<Iconography 41>
"C'est drôle tout d'même !... dire qu'on ne parle pas la même langue et qu'on s'entend à merveille !", déclare un soldat français, qui serre la main à un soldat anglais et à un soldat turc : [estampe]


Il y a aussi la variante **`db.get_or_404()` qui renvoie une erreur 404 si l'élément n'est pas trouvé**. C'est fort utile sur internet, quand les utilisateurices font n'importe quoi.

In [16]:
with app.app_context():
    # marche comme prévu
    icono_item = db.session.get(Iconography, 41)
    print(icono_item)

    # raise une erreur 404 Not Found
    icono_item = db.get_or_404(Iconography, 9999)
    print(icono_item)


<Iconography 41>


NotFound: 404 Not Found: The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.

## `.filter()`: équivalent de `WHERE` en SQL

### Utilisation de base

`.filter()` est équivalent de `WHERE` en SQL.

La requête SQL:

```py
SELECT * FROM iconography WHERE iconography.date_lower = 1850;
```

S'écrit:

In [ ]:
with app.app_context():
    query = db.select(Iconography).filter(Iconography.date_lower==1850)
    icono_1850 = db.session.execute(query).scalars().all()
print(icono_1850)
print("Nombre de ressources créées en 1850:", len(icono_1850))

**La syntaxe** est donc:

```py
db.select(NomDeTable).filter(mon_filtre)
```

**Un filtre est une comparaison avec les valeurs d'une colonne**:

In [ ]:
# comment interprétez vous ces requêtes ?
with app.app_context():
    db.select(Iconography).filter(Iconography.date_lower==1850)
    db.select(Iconography).filter(Iconography.date_upper>1900)
    db.select(Iconography).filter(Iconography.title.startswith("Palais Royal"))

En plus des opérateurs de comparaison vus ci-dessus (`==`, `>`, `<`, `>=`, `<=`, `startswith`), il existe de nombreux opérateurs, [visibles ici](https://docs.sqlalchemy.org/en/21/core/operators.html).

**On note que le reste de la requête ne change pas**: `db.select(Iconography)`, `db.session.execute()`, `.scalars().all()`.

### `and_()`, `or_()`, `not()_`: combiner les filtres

En SQL, `AND`, `OR`, `NOT` permettent de combiner plusieurs filtres (`condition1 et|ou|sauf condition2`). Leurs équivalents SQLAlchemy sont **les fonctions `and_()`, `or_()` et `not_()`**.

La requête SQL:

```sql
SELECT * FROM iconography WHERE iconography.date_lower>=1800 AND iconography.date_upper<=1900;
```

S'écrit:

In [ ]:
# ne pas oublier d'importer la fonction !
from sqlalchemy import and_

with app.app_context():
    query = db.select(Iconography).filter(and_(
        Iconography.date_lower >= 1800, 
        Iconography.date_upper <= 1900
    ))
    icono_19e_s = db.session.execute(query).scalars().all()
    print(icono_19e_s)

Comme en SQL, on peut combiner `and_()`, `or_()` et `not_()`. Voilà par exemple comment on affiche **toutes les ressources iconographiques produites pendant le 19e siècle qui ne représentent pas le Palais Royal** (pour simplifier, on exclut en fait toutes les oeuvres dont le titre est "Palais Royal"):

In [ ]:
# ne pas oublier d'importer la fonction !
from sqlalchemy import not_

with app.app_context():
    query = db.select(Iconography).filter(and_(
        Iconography.date_lower >= 1800, 
        Iconography.date_upper <= 1900,
        not_(Iconography.title == "Palais Royal")
    ))
    icono = db.session.execute(query).scalars().all()
    print(icono)

In [ ]:
# exercice:
# afficher toutes les ressources iconographiques qui représentent la "Galerie Vivienne" produites après le début du XXe siècle


## `.order_by()`: équivalent de `ORDER BY` en SQL

En SQL, `ORDER BY` permet d'ordonner les résultats:

```sql
SELECT * FROM iconography WHERE iconography.date_upper < 1900 ORDER BY iconography.date_lower
```

Avec SQLAlchemy, `.order_by` a le même rôle:

In [ ]:
with app.app_context():
    query = db.select(Iconography).filter(Iconography.date_upper<1900).order_by(Iconography.date_lower)
    icono_items = db.session.execute(query).scalars().all()

for item in icono_items:
    print(item.id, " : ", item.date_lower)

**La syntaxe est donc**:

```py
db.select(NomDeTable).order_by(NomDeTable.nom_de_colonne)
```

Comme en SQL, **`.order_by` s'ajoute après `.filter()`** (si notre requ6ete a un `.filter()`) et permet d'ordonner les résultats selon les données d'une colonne:

**On peut changer l'ordre**:
- `.order_by()` ordonne dans l'ordre ascendant par défaut
- `.desc()` permet d'ordonner dans l'ordre descendant:
    ```py
    .order_by( NomDeTable.nom_de_colonne.desc() )
    ```
- `.asc()` peut être utilisé de la même manière pour l'ordre ascendant, mais c'est inutile:
    ```py
    .order_by( NomDeTable.nom_de_colonne.asc() )
    ```

In [ ]:
with app.app_context():
    query = db.select(Iconography).filter(Iconography.date_upper<1900).order_by(Iconography.date_lower.desc())
    icono_items = db.session.execute(query).scalars().all()

for item in icono_items:
    print(item.id, " : ", item.date_lower)

## `.limit()`: équivalent de `LIMIT` en SQL

Si on ne veut pas afficher tous les résultats, on utilise une clause `LIMIT`. En SQL, cela s'écrit:

```sql
SELECT * FROM iconography WHERE iconography.date_lower > 1900 LIMIT 5;
```

Avec SQLAlchemy, on utilise `.limit()`:

In [ ]:
with app.app_context():
    query = db.select(Iconography).filter(Iconography.date_upper<1900).limit(5)
    icono_items = db.session.execute(query).scalars().all()
print(icono_items)

**La syntaxe est donc:**
```py
# `n` est le nombre de lignes qu'on veut retourner
db.select(NomDeTable).limit(n)
```

Comme en SQL, **`.limit()` s'ajoute après `.filter()` et `.order_by()`**.

## TLDR: construire une requête

L'ordre des opérations est le même en SQL et SQLAlchemy: `SELECT -> WHERE/FILTER -> ORDER BY -> LIMIT`  

```py
query = (
    db.select( NomDeTable )
    .filter( mon_filtre )
    .order_by( NomDeTable.nom_de_colonne )
    .limit( N )
```  


---

# Sélectionner les résultats à afficher: `.all()`, `.first()`, `.one()`

Reprennons la syntaxe qu'on a vu pour les requêtes SQLAlchemy:

```py
db.session.execute(query).scalars().all()
```

Au dessus, on a parlé de la partie `query`. Pour le moment, nos requêtes se terminent par `.scalars().all()`. On peut changer le `.all()` pour décider ce qu'on veut afficher:

- `.all()`: retourner tous les résultats dans une `list`
- `.one()`: la requête ne doit retourner qu'un résultat. Causer une erreur si 0 ou 2+ lignes sont retournées
- `.first()`: n'afficher que le premier résultat

***Note de performance**: c'est seulement quand on utilise `.all()`, `.first()`, `.one()` ou `.count()` qu'une requête s'exécute. Les requêtes SQL ont un coût de performance, et la traduction en Python faite par l'ORM aussi. Du coup:*
- *si vous n'avez pas besoin de toutes les lignes, utilisez `.first()` ou  `.limit(n)`, qu'on verra plus tard*
- *utilisez `.filter()` pour ne choisir que les données nécessaires*
- *n'exécutez les requêtes qu'une fois*



---

# Utiliser l'ORM dans une application

Avant de commencer à modifier l'appli pour utiliser l'ORM, essayez de modifier cette route pour **remplacer la requête SQL textuelle par une requête sur l'ORM**:

```py
@app.route("/iconographie")
def icono_index():
    icono_corpus = db.session.execute(text("SELECT * FROM iconography;")).all()
    return render_template("pages/icono_index.html", app_name=APP_NAME, icono_corpus=icono_corpus)
```

(la réponse est en dessous mais trichez pas svp)

In [ ]:
# votre code


On modifie l'appli pour y ajouter l'ORM.

## Le dossier `models/`

On ajoute à `app/` un dossier `models/` qui stockera tous nos modèles SQLAlchemy.

La structure de notre appli est maintenant:

```txt
app/
├── models/
│   └── data.py
├── routes/
├── statics/
├── templates/
└── utils/
```

**`app/models/data.py` stocke tous les modèles SQLAlchemy** "de données" (auxquelles on ajoutera plus tard dans d'autres fichiers les modèles relatives à la gestion d'utilisateurs):

```py
from sqlalchemy.orm import Mapped, mapped_column
from app.app import db

# comment lit-on chacun des attributs ci-dessous ?
class Iconography(db.Model):
    __tablename__ = "Iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    # NOTE: il reste à définir la relation avec la table `author`, mais on verra comment faire plus tard !
    id_author = ...
```

## Les routes

Dans `app/routes/generic.py`, **on met à jour nos routes pour faire des requêtes sur l'ORM**. Voici un extrait:

```py
@app.route("/iconographie/")
def icono_index():
    """
    index des ressources icono
    """
    # on affiche tout l'index icono: 
    query = db.select(Iconography)
    icono_corpus = db.session.execute( query ).scalars().all() 
    return render_template("pages/icono_index.html", app_name=APP_NAME, icono_corpus=icono_corpus)


@app.route("/iconographie/<int:id_icono>")
def icono_main(id_icono: int):
    """
    vue principale d'une ressource icono
    """
    icono_item = db.get_or_404(Iconography, id_icono)
    return render_template("pages/icono_main.html", app_name=APP_NAME, icono_item=icono_item)
```

## Les templates

Les templates sont aussi mises à jour pour une meilleure UI. Je ne m'attarde pas dessus, mais vous vouvez aller voir ce qu'on peut faire avec des templates plus complètes:

- [`icono_index.html`](./apps/s3/sqlalchemy_models/app/templates/pages/icono_index.html)
- [`icono_main.html`](./apps/s3/sqlalchemy_models/app/templates/pages/icono_main.html)

**Lancer l'application `sqlalchemy_models`**:
> ```py
> python apps/s3/sqlalchemy_models/main.py
> ```

---

# TLDR

Aujourd'hui, on a vu:

- comment faire des requêtes SQL `SELECT` en Python via SQLAlchemy
    - avec des requêtes textuelles (`text_()`) 
    - avec l'ORM
- comment modéliser une table en `class` avec l'ORM SQLAlchemy
- comment faire des requêtes `SELECT` via l'ORM
- comment intégrer les modèles et les requêtes ORM à une route d'une appli Flask. 

| SQLAlchemy                     | équivalent SQL     | rôle                     | où utiliser | 
|--------------------------------|--------------------|--------------------------|-------------|
| `db.session.execute()`          | -                  | exécuter la requête SQL en argument |  |
| `db.session.get()`             | `WHERE id = n`    | récupérer une ligne par son ID      |  |
| `db.session.get_or_404()`      | `WHERE id = n`    | pareil, mais lance une erreur 404 si l'objet n'est pas trouvé | |
|  `db.select()`                 | `SELECT`           | commencer une requête SQL | |
| `.filter()`                    | `WHERE`            | filtrer les données à afficher | après `.select()` |
| `and_()`, `or_()`, `not_()`    | `AND`, `OR`, `NOT` | combiner plusieurs filtres | dans `.filter()`  |
| `.order_by()`                  | `ORDER BY          | ordonner les résultats | après `.filter()` |
| `.limit()`                     | `LIMIT`            | limiter le nombre de résultats affichés | apreès `.order_by()` |
| `.scalars()`                   | -                  | simplifier la structure de la réponse | après `.execute()` |
| `.all()`, `.first()`, `.one()` | -                  | exécuter une requête et choisir combien de résultats retourner | à la fin d'une requête |